# Ollama qua REST API thô

`llmkit.chat` gói gọn một lời gọi Ollama trong một dòng. Bài này mở gói đó ra: bạn tự dựng payload `/api/chat`,
đọc **stream NDJSON** từng mảnh, tính tốc độ từ các trường thời gian Ollama trả về, đọc thông tin model
(context length, quantization, giấy phép) từ `/api/show`, viết Modelfile, rồi đo trên server thật: thời gian tới
token đầu (TTFT), token/s, và cái bẫy **`num_ctx`** cắt prompt dài mà không báo lỗi.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell `explore` gọi Ollama thật (cần `ollama serve` với `qwen2.5:3b`), cần bộ `wikipedia-vi` cho prompt dài.
  Server có thể dùng chung: các cell chỉ gọi khoảng 12 lần.
- Chấm: `uv run pytest tracks/ai-engineer/modules/09-llms-and-open-models/exercises/intermediate/02-ollama-rest -q`

In [ ]:
import json
from collections.abc import Iterable

## 1. Payload và stream

`chat_payload(model, messages, *, stream=True, temperature=None, top_p=None, top_k=None, num_ctx=None,
num_predict=None, seed=None, keep_alive=None, format=None) -> dict`:

- Luôn có `model`, `messages`, `stream`.
- Các tham số sampling và `num_ctx`, `num_predict`, `seed` khác `None` nằm trong dict con `options`, đúng tên
  tham số. Không có option nào thì **không** có khoá `options`.
- `keep_alive` (vd. `"5m"`, `0`) và `format` (`"json"` hoặc một JSON schema) khác `None` thì ở cấp ngoài cùng.

`parse_stream(lines) -> tuple[str, dict]`: `lines` là các dòng NDJSON (kiểu `str` hoặc `bytes`) của
`POST /api/chat` khi `stream: true`. Bỏ dòng rỗng. Nối mọi `message.content` thành câu trả lời; trả kèm object
cuối có `"done": true` (chứa thống kê). Dòng có khoá `"error"`: `RuntimeError` mang thông điệp lỗi. Hết dòng mà
chưa gặp `done: true`: `RuntimeError`.

In [ ]:
def chat_payload(model: str, messages: list[dict], *, stream: bool = True, temperature: float | None = None,
                 top_p: float | None = None, top_k: int | None = None, num_ctx: int | None = None,
                 num_predict: int | None = None, seed: int | None = None, keep_alive: str | int | None = None,
                 format: str | dict | None = None) -> dict:
    """Body JSON cho POST /api/chat của Ollama."""
    # TODO
    raise NotImplementedError


def parse_stream(lines: Iterable[str | bytes]) -> tuple[str, dict]:
    """Nối content của các mảnh NDJSON; trả (văn bản, object cuối có done=true)."""
    # TODO
    raise NotImplementedError

## 2. Tốc độ, thông tin model, Modelfile

`speed_stats(final) -> dict` từ object cuối của stream (mọi thời gian Ollama trả tính bằng **nano giây**):

| Khoá | Cách tính |
| --- | --- |
| `load_s` | `load_duration` (giây) |
| `prompt_tokens`, `gen_tokens` | `prompt_eval_count`, `eval_count` |
| `prompt_tps`, `gen_tps` | số token chia thời gian tương ứng (`prompt_eval_duration`, `eval_duration`) tính bằng giây |
| `total_s` | `total_duration` (giây) |

Trường thiếu coi là 0 (một số phiên bản Ollama bỏ trường khi giá trị bằng 0); thời gian bằng 0 thì tốc độ là `0.0`.

`model_facts(show) -> dict` từ JSON của `POST /api/show`: `architecture` (`model_info["general.architecture"]`),
`parameter_count` (`general.parameter_count`), `context_length` (`model_info["<architecture>.context_length"]`),
`quantization` (`details["quantization_level"]`), `license`: tên giấy phép `general.license.name` nếu có (GGUF ghi
`general.license` là `"other"` cho giấy phép không chuẩn và để tên thật ở khoá này), không thì `general.license`, thiếu
cả hai thì `None`.

`render_modelfile(base, system=None, parameters=None) -> str`: dòng `FROM <base>`, rồi mỗi tham số một dòng
`PARAMETER <tên> <giá trị>` theo thứ tự dict (giá trị là list thì mỗi phần tử một dòng, như `stop`), rồi nếu có
system: `SYSTEM """<system>"""`. Mỗi dòng kết thúc bằng `\n`.


In [ ]:
def speed_stats(final: dict) -> dict:
    """Tốc độ và thời gian từ object done=true của Ollama (ns -> s)."""
    # TODO
    raise NotImplementedError


def model_facts(show: dict) -> dict:
    """Kiến trúc, số tham số, context length, quantization, giấy phép từ /api/show."""
    # TODO
    raise NotImplementedError


def render_modelfile(base: str, system: str | None = None, parameters: dict | None = None) -> str:
    """Modelfile: FROM, các PARAMETER, SYSTEM."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: server có gì: model đã tải (/api/tags), model đang nạp trong RAM/VRAM (/api/ps), thông tin từng model
import os
import sys
import time
from pathlib import Path

import httpx

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import llmkit
from data.download_dataset import ensure

HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MODEL = "qwen2.5:3b"
http = httpx.Client(base_url=HOST, timeout=1800)
print("Ollama", http.get("/api/version").json()["version"])
for m in http.get("/api/tags").json()["models"]:
    print(f"  {m['name']:24s} {m['size'] / 1e9:5.2f} GB  {m['details']['parameter_size']:>7s}  "
          f"{m['details']['quantization_level']}")
for name in ("qwen2.5:3b", "qwen2.5:7b"):
    show = http.post("/api/show", json={"model": name})
    print(name, model_facts(show.json()) if show.status_code == 200 else "chưa pull")


In [ ]:
# explore: nạp model trước (messages rỗng chỉ nạp, không sinh), rồi stream một câu trả lời: TTFT và token/s
t0 = time.perf_counter()
http.post("/api/chat", json={"model": MODEL, "messages": [], "stream": False})
print(f"nạp {MODEL}: {time.perf_counter() - t0:.2f} s (gần 0 nếu model đã nằm sẵn trong bộ nhớ)")

messages = [{"role": "user", "content": "Giải thích ngắn gọn trong 3 câu: knowledge cut-off của LLM là gì?"}]
payload = chat_payload(MODEL, messages, temperature=0, seed=42, num_predict=120)
t0, ttft, lines = time.perf_counter(), None, []
with http.stream("POST", "/api/chat", json=payload) as response:
    for line in response.iter_lines():
        if ttft is None and line.strip() and json.loads(line).get("message", {}).get("content"):
            ttft = time.perf_counter() - t0
        lines.append(line)
wall = time.perf_counter() - t0
text, final = parse_stream(lines)
stats = speed_stats(final)
print(text)
print(f"\n{len(lines)} dòng NDJSON, done_reason {final.get('done_reason')!r}, TTFT {ttft:.3f} s, tổng {wall:.2f} s")
print({k: round(v, 3) for k, v in stats.items()})
for m in http.get("/api/ps").json()["models"]:
    print(f"/api/ps: {m['name']} nạp {m['size'] / 1e9:.2f} GB, trên GPU {m['size_vram'] / 1e9:.2f} GB, "
          f"context_length {m.get('context_length')}")


In [ ]:
# explore: temperature 0 có tất định không? Cùng câu hỏi gọi lại 4 lần, qua llmkit, và qua /v1/chat/completions
again = [http.post("/api/chat", json=chat_payload(MODEL, messages, stream=False, temperature=0, seed=42,
                                                  num_predict=120)).json()["message"]["content"] for _ in range(4)]
print("4 lần gọi lại /api/chat, giống hệt lần stream:", [a == text for a in again])
print("4 lần gọi lại giống nhau:", len(set(again)) == 1)
same = llmkit.chat(messages, model=MODEL, temperature=0)
print("llmkit.chat (không seed, không giới hạn num_predict), 200 ký tự đầu giống lần gọi lại:", same[:200] == again[-1][:200])
oa = http.post("/v1/chat/completions", json={"model": MODEL, "messages": messages, "temperature": 0, "seed": 42,
                                             "max_tokens": 120}).json()
print("/v1/chat/completions:", oa["choices"][0]["message"]["content"][:120], "...")
print("usage:", oa["usage"])


In [ ]:
# explore: bẫy num_ctx: prompt khoảng 7 600 token, dữ kiện cần nhớ nằm ở ĐẦU prompt
import pyarrow.parquet as pq

WIKI = ensure("wikipedia-vi")
batch = next(pq.ParquetFile(WIKI / "20231101.vi" / "train-00001-of-00004.parquet").iter_batches(batch_size=400, columns=["text"]))
filler, total = [], 0
for t in batch.column("text").to_pylist():
    filler.append(t[:2000])
    total += len(t[:2000])
    if total > 20_000:
        break
fact = "Ghi chú quan trọng: mã số của kho lưu trữ là 7429."
long_messages = [{"role": "user", "content": fact + "\n\n" + "\n\n".join(filler)
                  + "\n\nCâu hỏi: theo ghi chú quan trọng ở đầu văn bản, mã số của kho lưu trữ là gì? Chỉ trả lời con số."}]
print(f"{len(long_messages[0]['content']):,} ký tự")
for num_ctx in (None, 8192):
    body = chat_payload(MODEL, long_messages, stream=False, temperature=0, seed=1, num_predict=16, num_ctx=num_ctx)
    out = http.post("/api/chat", json=body).json()
    s = speed_stats(out)
    print(f"num_ctx={num_ctx or 'mặc định'}: prompt_eval_count {s['prompt_tokens']}, "
          f"trả lời {out['message']['content'].strip()!r}, đúng: {'7429' in out['message']['content']}, "
          f"đọc prompt {s['prompt_tps']:.0f} token/s, tổng {s['total_s']:.1f} s (gồm nạp lại {s['load_s']:.1f} s)")
_ = http.post("/api/chat", json={"model": MODEL, "messages": [], "keep_alive": 0})


In [ ]:
# explore: Modelfile: model con "ae09-tro-ly" từ qwen2.5:3b (dùng chung trọng số, chỉ thêm system và tham số), rồi xoá
system = "Bạn là trợ lý tiếng Việt. Luôn trả lời đúng một câu, không quá 25 từ."
params = {"temperature": 0.2, "num_ctx": 4096, "stop": ["<|im_end|>", "<|endoftext|>"]}
print(render_modelfile(MODEL, system, params))
r = http.post("/api/create", json={"model": "ae09-tro-ly", "from": MODEL, "system": system, "parameters": params,
                                   "stream": False})
print("create:", r.json())
for question in ("Hugging Face là gì?", "Ollama là gì?"):
    ans = http.post("/api/chat", json=chat_payload("ae09-tro-ly", [{"role": "user", "content": question}],
                                                   stream=False, seed=1)).json()
    print(f"ae09-tro-ly | {question} -> {ans['message']['content']!r}")
_ = http.post("/api/chat", json={"model": "ae09-tro-ly", "messages": [], "keep_alive": 0})
print("xoá:", http.request("DELETE", "/api/delete", json={"model": "ae09-tro-ly"}).status_code)


## Đọc kết quả

- TTFT so với thời gian sinh cả câu trả lời: người dùng chờ bao lâu mới thấy chữ đầu tiên nếu không stream? Lần nạp
  model đầu tiên tốn bao lâu, và vì sao `keep_alive` quan trọng với độ trễ?
- Tốc độ đọc prompt (`prompt_tps`) và tốc độ sinh (`gen_tps`) chênh nhau bao nhiêu lần? Vì sao?
- `/api/ps` cho thấy model nằm hết trên GPU hay bị chia GPU/CPU? `context_length` đang nạp là bao nhiêu, so với
  context length mà model hỗ trợ trong `/api/show`?
- `temperature` 0 cộng `seed` cố định: các lần gọi có cho cùng một chuỗi không? Lần đầu và các lần sau khác nhau ở đâu?
- Với `num_ctx` mặc định, `prompt_eval_count` là bao nhiêu và model có trả lời đúng mã số không? Ollama có báo lỗi
  không?
- Model con `ae09-tro-ly` trả lời có đúng sự thật, có đúng tiếng Việt không? Điều đó nói gì về model 3B và về system
  prompt ép trả lời thật ngắn?

Sau đó làm phần trắc nghiệm của bài trong app.
